In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [3]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/tourism_dataset_5000.csv')

Mounted at /content/drive


In [4]:
# Pré-processamento dos Dados
# Separando features e target
X = df.drop('Satisfaction', axis=1)
y = df['Satisfaction']

In [5]:
# Identificando colunas numéricas e categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

In [6]:
# Criando transformers para pré-processamento
numeric_transformer = Pipeline(steps=[
   ('imputer', SimpleImputer(strategy='median')),  # Imputação de valores faltantes
   ('scaler', StandardScaler())  # Escalonamento
])


categorical_transformer = Pipeline(steps=[
   ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Imputação de valores faltantes
   ('onehot', OneHotEncoder(handle_unknown='ignore'))  # Codificação one-hot
])

In [7]:
# Combinando transformers em um ColumnTransformer
preprocessor = ColumnTransformer(
  transformers=[
      ('num', numeric_transformer, numeric_features),
      ('cat', categorical_transformer, categorical_features)
  ]
)

In [8]:
# Divisão dos Dados em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Seleção do Modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)

In [10]:
# Criando a Pipeline completa
pipeline = Pipeline(steps=[
   ('preprocessor', preprocessor),
   ('classifier', model)
])

In [11]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Tourist ID', 'Age', 'Preferred Tour Duration', 'Tour Duration',
       'Route ID', 'Tourist Rating', 'System Response Time',
       'Recommendation Accuracy', 'VR Experience Quality'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Interests', 'Site Name', 'Sites Visited'], dtype='object'))])),
                ('classifier', RandomForestRegressor(random_state=42))])

In [12]:
# Avaliação do Modelo
y_pred = pipeline.predict(X_test)

# Métricas de Regressão
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}')
print(f'R²: {r2_score(y_test, y_pred)}')

MSE: 0.0
MAE: 0.0
RMSE: 0.0
R²: 1.0


In [23]:
# Seleciona um registro do dataset
novo_turista = X.iloc[[0]].copy()

# Altera alguns valores
novo_turista['Duration'] = 10
novo_turista['Travel_Cost'] = 6000
novo_turista['Tourist Rating'] = 5

# Faz a previsão
previsao = pipeline.predict(novo_turista)

# Transforma os dados do turista em uma tabela vertical
resultado = novo_turista.T.reset_index()

# Define os nomes das colunas
resultado.columns = ['Variável', 'Valor']

# Adiciona a previsão ao final da tabela
resultado.loc[len(resultado)] = [
    'Satisfação prevista',
    f'{previsao[0]:.2f}'
]

# Exibe a tabela
display(resultado)

,Variável,Valor
0,Tourist ID,1
1,Age,48
2,Interests,"['Architecture', 'Art', 'History']"
3,Preferred Tour Duration,5
4,Accessibility,False
5,Site Name,Eiffel Tower
6,Sites Visited,"['Eiffel Tower', 'Great Wall of China', 'Taj M..."
7,Tour Duration,7
8,Route ID,1000
9,Tourist Rating,5
